In [ ]:
import sys, os, json, warnings, time, pickle
from pathlib import Path

# Ensure repo root is on path whether we launch from notebooks/ or root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mlflow
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
)

# ── MLflow tracking ──────────────────────────────────────────────────────────
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('05_Pipeline_Walkthrough')

# ── Paths ────────────────────────────────────────────────────────────────────
CHECKPOINTS = Path('artifacts/checkpoints')
METRICS_DIR = Path('artifacts/metrics')
SHAP_DIR    = Path('artifacts/shap')
TREE_DIR    = Path('data/processed/tree_models')

print('Working directory:', os.getcwd())
print('MLflow experiment: 05_Pipeline_Walkthrough')
print('Python version:', sys.version.split()[0])

---
## 5.1 — Preprocessing Audit

Validates that the preprocessing pipeline produces time-safe train/val splits with no future leakage.

In [ ]:
# ── Run 5a: Preprocessing audit ───────────────────────────────────────────────
with mlflow.start_run(run_name='5a_preprocessing') as run:

    # Load processed datasets
    try:
        train_df = pd.read_parquet(f'{TREE_DIR}/train.parquet')
        val_df   = pd.read_parquet(f'{TREE_DIR}/val.parquet')
    except FileNotFoundError:
        print('Processed data not found — run Exp 1/2 first to generate it.')
        mlflow.set_tag('status', 'skipped_no_data')
        raise

    n_train     = len(train_df)
    n_val       = len(val_df)
    fraud_rate_train = train_df['isFraud'].mean()
    fraud_rate_val   = val_df['isFraud'].mean()
    n_features  = train_df.shape[1]

    # Time-split integrity check: latest train DT < earliest val DT
    max_train_dt = train_df['TransactionDT'].max()
    min_val_dt   = val_df['TransactionDT'].min()
    time_split_ok = bool(max_train_dt < min_val_dt)

    # NaN audit after preprocessing
    numeric_train = train_df.select_dtypes(include=[np.number])
    nan_pct_train = numeric_train.isna().sum().sum() / numeric_train.size * 100
    nan_pct_val   = val_df.select_dtypes(include=[np.number]).isna().sum().sum() / \
                    val_df.select_dtypes(include=[np.number]).size * 100

    mlflow.log_params({
        'dataset':         'IEEE-CIS Fraud Detection',
        'time_split_col':  'TransactionDT',
        'split_ratio':     '75/25',
    })
    mlflow.log_metrics({
        'n_train':            n_train,
        'n_val':              n_val,
        'n_features':         n_features,
        'fraud_rate_train':   round(fraud_rate_train, 5),
        'fraud_rate_val':     round(fraud_rate_val, 5),
        'time_split_valid':   int(time_split_ok),
        'nan_pct_train':      round(nan_pct_train, 3),
        'nan_pct_val':        round(nan_pct_val, 3),
        'max_train_dt':       int(max_train_dt),
        'min_val_dt':         int(min_val_dt),
    })
    mlflow.set_tags({
        'stage':          'preprocessing',
        'time_split_ok':  str(time_split_ok),
    })

print(f'Train: {n_train:,} rows | Val: {n_val:,} rows | Features: {n_features}')
print(f'Fraud rate — Train: {fraud_rate_train:.3%} | Val: {fraud_rate_val:.3%}')
print(f'Time-split integrity: {"✅ PASS" if time_split_ok else "❌ FAIL — DATA LEAKAGE!"}')
print(f'NaN % (numeric) — Train: {nan_pct_train:.2f}% | Val: {nan_pct_val:.2f}%')
print(f'max(train DT)={max_train_dt:,}  min(val DT)={min_val_dt:,}')

---
## 5.2 — Feature Engineering Audit

Confirms all feature groups are present and inspects their distributions.

In [ ]:
# ── Run 5b: Feature engineering audit ────────────────────────────────────────
FEATURE_GROUPS = {
    'time':      ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_late_night'],
    'email':     ['email_match', 'P_email_is_free', 'email_trusted', 'P_email_tld_freq'],
    'device':    ['device_mobile', 'device_os_encoded', 'device_browser_encoded'],
    'c_cols':    ['C1_C2_ratio', 'C5_C6_ratio', 'C6_C11_ratio', 'C3_flag'],
    'd_cols':    ['D1_norm', 'D2_norm'],
    'm_cols':    ['M_match_count', 'M_mismatch_count', 'M_nan_count'],
    'velocity':  ['uid_tx_1h', 'uid_tx_24h', 'uid_total_amt_24h'],
    'uid_agg':   ['uid_amt_mean', 'uid_amt_std'],
    'amount':    ['TransactionAmt_log', 'amt_zscore'],
    'freq_enc':  ['card1_freq', 'P_emaildomain_freq'],
    'vae':       ['vae_anomaly_score'],
}

with mlflow.start_run(run_name='5b_feature_engineering') as run:
    group_coverage = {}
    missing_features = []

    for group, cols in FEATURE_GROUPS.items():
        present = [c for c in cols if c in train_df.columns]
        pct = len(present) / len(cols) * 100
        group_coverage[group] = pct
        if len(present) < len(cols):
            missing_features.extend([c for c in cols if c not in train_df.columns])
        mlflow.log_metric(f'coverage_{group}_pct', round(pct, 1))

    total_coverage = np.mean(list(group_coverage.values()))
    mlflow.log_metric('total_feature_coverage_pct', round(total_coverage, 1))
    mlflow.log_metric('n_missing_features', len(missing_features))
    mlflow.set_tag('stage', 'feature_engineering')

    if missing_features:
        mlflow.set_tag('missing_features', ', '.join(missing_features[:10]))

# Print coverage table
print(f'{'Group':<15} {'Coverage':>10}  Status')
print('-' * 40)
for group, pct in group_coverage.items():
    status = '✅' if pct == 100 else '⚠️ partial'
    print(f'{group:<15} {pct:>9.1f}%  {status}')
print(f'{"OVERALL":<15} {total_coverage:>9.1f}%')

if missing_features:
    print(f'\n Missing: {missing_features}')

In [ ]:
# Feature group coverage bar chart
fig, ax = plt.subplots(figsize=(12, 4))
groups  = list(group_coverage.keys())
values  = list(group_coverage.values())
colors  = ['#2ecc71' if v == 100 else '#e67e22' for v in values]

bars = ax.bar(groups, values, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, 115)
ax.set_ylabel('Feature Coverage (%)')
ax.set_title('Feature Group Coverage — Training Dataset', fontsize=13, fontweight='bold')
ax.axhline(100, color='#e74c3c', ls='--', lw=1, label='100% target')
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(str(SHAP_DIR / 'exp05_feature_coverage.png'), dpi=150, bbox_inches='tight')
plt.show()

with mlflow.start_run(run_name='5b_feature_engineering', nested=True):
    mlflow.log_artifact(str(SHAP_DIR / 'exp05_feature_coverage.png'))

---
## 5.3 — Model Artifact Audit

Confirms all trained model checkpoints exist and logs their file sizes.

In [ ]:
# ── Run 5c: Model artifact audit ─────────────────────────────────────────────
EXPECTED_ARTIFACTS = {
    # Tree models
    'xgb_final.pkl':           'XGBoost (final retrained)',
    'lgbm_final.pkl':          'LightGBM (final retrained)',
    'catboost_final.cbm':      'CatBoost (final retrained)',
    # Deep learning
    'bilstm_final.pt':         'BiLSTM (final)',
    'vae_best.pt':             'VAE (best)',
    'gnn_best.pt':             'GNN (best)',
    # Stacking
    'meta_learner.pkl':        'Stacking meta-learner',
    # Encoders
    'feature_encoders.pkl':    'Feature encoders',
    'bilstm_platt_scaler.pkl': 'BiLSTM Platt scaler',
    'vae_scaler.pkl':          'VAE scaler',
    # Stats
    'cohort_priors.json':      'Cohort priors (cold-start)',
    'vae_score_stats.npy':     'VAE score statistics',
    'xgb_feature_cols.csv':    'XGBoost feature column list',
    # GNN embeddings
    'gnn_card_embeddings.npy': 'GNN card embeddings',
    'gnn_card_map.pkl':        'GNN card → node map',
}

audit_rows = []
total_size_mb = 0.0

for filename, description in EXPECTED_ARTIFACTS.items():
    path = CHECKPOINTS / filename
    exists = path.exists()
    size_mb = path.stat().st_size / 1e6 if exists else 0.0
    total_size_mb += size_mb
    audit_rows.append({
        'Artifact': filename,
        'Description': description,
        'Exists': '✅' if exists else '❌',
        'Size (MB)': f'{size_mb:.1f}',
    })

audit_df = pd.DataFrame(audit_rows)
n_present = (audit_df['Exists'] == '✅').sum()
n_total   = len(audit_df)

with mlflow.start_run(run_name='5c_model_artifacts') as run:
    mlflow.log_metrics({
        'artifacts_present':   n_present,
        'artifacts_total':     n_total,
        'artifacts_pct':       round(n_present / n_total * 100, 1),
        'total_checkpoint_mb': round(total_size_mb, 1),
    })
    mlflow.set_tags({
        'stage':          'model_audit',
        'all_present':    str(n_present == n_total),
    })

print(audit_df.to_string(index=False))
print(f'\n{n_present}/{n_total} artifacts present | Total size: {total_size_mb:.1f} MB')

---
## 5.4 — Inference Engine Demo (Single Transaction)

Runs a single synthetic transaction through the **full production pipeline** — the same path as `POST /score` in production.

In [ ]:
# ── Synthetic transaction (mirrors IEEE-CIS schema) ───────────────────────────
sample_transactions = {
    'legitimate': {
        'TransactionID': 9000001,
        'TransactionDT': 86400,
        'TransactionAmt': 49.0,
        'ProductCD': 'W',
        'card1': 'card_legit_001', 'card2': '321.0', 'card3': '150.0',
        'card4': 'visa', 'card5': '226.0', 'card6': 'debit',
        'addr1': '330', 'addr2': '87.0',
        'P_emaildomain': 'gmail.com', 'R_emaildomain': 'gmail.com',
        'DeviceType': 'desktop', 'DeviceInfo': 'Windows',
        'id_31': 'chrome 72.0',
        'C1': 1.0, 'C2': 1.0, 'C3': 0.0, 'C4': 0.0,
        'C5': 0.0, 'C6': 1.0, 'C7': 0.0, 'C8': 0.0,
        'C9': 1.0, 'C10': 0.0, 'C11': 2.0, 'C12': 0.0, 'C13': 23.0, 'C14': 1.0,
        'D1': 14.0, 'D2': 14.0,
        'M1': 'T', 'M2': 'T', 'M3': 'T', 'M4': 'M0',
        'M5': 'F', 'M6': 'T', 'M7': 'F', 'M8': 'F', 'M9': 'T',
    },
    'suspicious': {
        'TransactionID': 9000002,
        'TransactionDT': 86400,
        'TransactionAmt': 999.99,       # high amount
        'ProductCD': 'C',
        'card1': 'card_susp_999', 'card2': '111.0', 'card3': '185.0',
        'card4': 'mastercard', 'card5': '102.0', 'card6': 'credit',
        'addr1': '000', 'addr2': '60.0',
        'P_emaildomain': 'protonmail.com', 'R_emaildomain': 'gmail.com',
        'DeviceType': 'mobile', 'DeviceInfo': 'Android',
        'id_31': 'samsung browser 7.0',
        'C1': 8.0, 'C2': 8.0, 'C3': 3.0, 'C4': 3.0,
        'C5': 3.0, 'C6': 8.0, 'C7': 3.0, 'C8': 3.0,
        'C9': 0.0, 'C10': 3.0, 'C11': 8.0, 'C12': 3.0, 'C13': 1.0, 'C14': 0.0,
        'D1': 0.0, 'D2': 0.0,                    # brand new card usage
        'M1': 'F', 'M2': 'F', 'M3': 'F', 'M4': 'M2',
        'M5': 'T', 'M6': 'F', 'M7': 'T', 'M8': 'T', 'M9': 'F',
    },
}

In [ ]:
# ── Run 5d: Single transaction inference demo ─────────────────────────────────
# This cell runs feature engineering directly on the synthetic transactions
# (without the Redis/production wiring) so the notebook works standalone.
from src.preprocessing.missing_values        import encode_m_columns, fill_nan_for_trees
from src.feature_engineering.time_features   import build_time_features
from src.feature_engineering.email_features  import build_email_features
from src.feature_engineering.device_features import (
    build_device_features, get_device_encoders, apply_device_encoders,
)
from src.feature_engineering.d_features      import build_d_features
from src.feature_engineering.c_features      import build_c_features
from src.feature_engineering.m_features      import build_m_features
from src.feature_engineering.amount_features import build_amount_features

results = {}

with mlflow.start_run(run_name='5d_inference_single') as run:
    mlflow.set_tag('stage', 'inference_demo')

    for label, tx in sample_transactions.items():
        t0 = time.time()

        df = pd.DataFrame([tx.copy()])

        # Full feature engineering chain (same order as production predictor)
        df = encode_m_columns(df, nan_value=-1)
        df = build_time_features(df)
        df = build_d_features(df, train_idx=None)
        df = build_c_features(df, train_idx=None)
        df = build_m_features(df, m_cols=None, train_idx=None)
        df = build_email_features(df)
        df = build_device_features(df)
        try:
            df = build_amount_features(df)
        except Exception:
            df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

        device_encs = get_device_encoders(df)
        df = apply_device_encoders(df, device_encs)
        fill_nan_for_trees(df, exclude_cols=[])

        fe_time_ms = (time.time() - t0) * 1000

        # Load trained models and run stacking
        try:
            with open(CHECKPOINTS / 'feature_encoders.pkl', 'rb') as f:
                encoders = pickle.load(f)
            with open(CHECKPOINTS / 'meta_learner.pkl', 'rb') as f:
                meta_artifacts = pickle.load(f)

            feat_df  = pd.read_csv(CHECKPOINTS / 'xgb_feature_cols.csv')
            feat_cols = feat_df['col'].tolist()

            freq_maps = encoders.get('freq_maps', {})
            for col, fmap in freq_maps.items():
                if col in df.columns:
                    df[f'{col}_freq'] = df[col].map(fmap).fillna(0.0)

            X = df.reindex(columns=feat_cols).fillna(-999).values

            import xgboost as xgb, lightgbm as lgb
            with open(CHECKPOINTS / 'xgb_final.pkl', 'rb') as f:
                xgb_model  = pickle.load(f)
            with open(CHECKPOINTS / 'lgbm_final.pkl', 'rb') as f:
                lgbm_model = pickle.load(f)

            xgb_prob  = float(xgb_model.predict_proba(X)[:, 1][0])
            lgbm_prob = float(lgbm_model.predict_proba(X)[:, 1][0])

            # CatBoost
            from catboost import CatBoostClassifier, Pool
            cb_model = CatBoostClassifier()
            cb_model.load_model(str(CHECKPOINTS / 'catboost_final.cbm'))
            cat_cols = encoders.get('cat_cols', [])
            cb_feats = cb_model.feature_names_
            X_cb = df.reindex(columns=cb_feats)
            for c in cat_cols:
                if c in X_cb.columns:
                    X_cb[c] = X_cb[c].astype(str).fillna('unknown')
            cat_idx = [cb_feats.index(c) for c in cat_cols if c in cb_feats]
            cb_prob = float(cb_model.predict_proba(Pool(X_cb, cat_features=cat_idx))[:, 1][0])

            # Stacking meta-learner
            from src.models.stacking.meta_learner import predict_meta_learner, apply_decision_thresholds
            probs_dict = {
                'xgb_prob':      np.array([xgb_prob]),
                'lgbm_prob':     np.array([lgbm_prob]),
                'catboost_prob': np.array([cb_prob]),
            }
            stacking_score = float(predict_meta_learner(
                meta_artifacts['model'], meta_artifacts['scaler'], probs_dict
            )[0])

            # Cold-start fusion (no BiLSTM history in this demo → use stacking)
            from src.inference.predict import _get_final_score
            tx_count   = 0   # new user in this demo
            final_prob = _get_final_score(stacking_score, 0.5, tx_count)

            # Decision thresholds
            decision_df = apply_decision_thresholds(np.array([final_prob]))
            decision    = decision_df.iloc[0]['decision']

            total_time_ms = (time.time() - t0) * 1000

            results[label] = {
                'xgb':           xgb_prob,
                'lgbm':          lgbm_prob,
                'catboost':      cb_prob,
                'stacking':      stacking_score,
                'final_prob':    final_prob,
                'decision':      decision,
                'latency_ms':    round(total_time_ms, 1),
                'fe_time_ms':    round(fe_time_ms, 1),
            }

            mlflow.log_metrics({
                f'{label}_xgb_prob':      round(xgb_prob, 5),
                f'{label}_lgbm_prob':     round(lgbm_prob, 5),
                f'{label}_catboost_prob': round(cb_prob, 5),
                f'{label}_stacking':      round(stacking_score, 5),
                f'{label}_final_prob':    round(final_prob, 5),
                f'{label}_latency_ms':    round(total_time_ms, 1),
            })
            mlflow.set_tag(f'{label}_decision', decision)

            print(f'\n[{label.upper()}]')
            print(f'  XGB:      {xgb_prob:.4f}  |  LGBM: {lgbm_prob:.4f}  |  CB: {cb_prob:.4f}')
            print(f'  Stacking: {stacking_score:.4f}  |  Final: {final_prob:.4f}')
            print(f'  Decision: {decision.upper()}  |  Latency: {total_time_ms:.1f}ms')

        except FileNotFoundError as e:
            print(f'⚠️  Checkpoint missing ({e}). Run Exp 1-4 first to generate model artifacts.')
            mlflow.set_tag('status', 'skipped_no_checkpoints')

In [ ]:
# Visualise per-model scores for both transactions
if results:
    model_keys = ['xgb', 'lgbm', 'catboost', 'stacking', 'final_prob']
    labels_map = ['XGBoost', 'LightGBM', 'CatBoost', 'Stacking', 'Final']
    x = np.arange(len(model_keys))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = {'legitimate': '#2ecc71', 'suspicious': '#e74c3c'}

    for i, (label, color) in enumerate(colors.items()):
        if label not in results:
            continue
        vals = [results[label].get(k, 0) for k in model_keys]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=label.capitalize(),
                       color=color, alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.axhline(0.5, color='gray', ls='--', lw=1, label='0.5 threshold')
    ax.set_xticks(x); ax.set_xticklabels(labels_map, fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Fraud Probability')
    ax.set_title('Per-Model Scores: Legitimate vs Suspicious Transaction', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(str(SHAP_DIR / 'exp05_inference_demo.png'), dpi=150, bbox_inches='tight')
    plt.show()

    with mlflow.start_run(run_name='5d_inference_single', nested=True):
        mlflow.log_artifact(str(SHAP_DIR / 'exp05_inference_demo.png'))

---
## 5.5 — Batch Inference Benchmark

Measures throughput on the validation set using the **vectorised batch predictor**.

In [ ]:
# ── Run 5e: Batch inference benchmark on val set ──────────────────────────────
# Load trained models once
try:
    EXCLUDE = ['TransactionID', 'isFraud', 'uid1', 'uid2', 'uid3', 'uid_magic',
               'card4', 'card6', 'ProductCD', 'DeviceType', 'P_emaildomain',
               'R_emaildomain', 'DeviceInfo', 'device_os', 'device_browser',
               'P_email_tld', 'R_email_tld']

    feat_df   = pd.read_csv(CHECKPOINTS / 'xgb_feature_cols.csv')
    feat_cols = feat_df['col'].tolist()

    with open(CHECKPOINTS / 'xgb_final.pkl', 'rb') as f:
        xgb_model = pickle.load(f)
    with open(CHECKPOINTS / 'lgbm_final.pkl', 'rb') as f:
        lgbm_model = pickle.load(f)
    with open(CHECKPOINTS / 'meta_learner.pkl', 'rb') as f:
        meta_arts = pickle.load(f)

    X_val_arr = val_df.reindex(columns=feat_cols).fillna(-999).values
    y_val     = val_df['isFraud'].values

    # Benchmark: vectorised batch inference
    import xgboost as xgb, lightgbm as lgb
    from src.models.stacking.meta_learner import predict_meta_learner, apply_decision_thresholds

    t0 = time.time()
    xgb_probs  = xgb_model.predict_proba(X_val_arr)[:, 1]
    lgbm_probs = lgbm_model.predict_proba(X_val_arr)[:, 1]

    # CatBoost batch
    cb_model = CatBoostClassifier()
    cb_model.load_model(str(CHECKPOINTS / 'catboost_final.cbm'))
    with open(CHECKPOINTS / 'feature_encoders.pkl', 'rb') as f:
        encoders = pickle.load(f)
    cat_cols = encoders.get('cat_cols', [])
    cb_feats = cb_model.feature_names_
    X_cb = val_df.reindex(columns=cb_feats)
    for c in cat_cols:
        if c in X_cb.columns:
            X_cb[c] = X_cb[c].astype(str).fillna('unknown')
    cat_idx = [cb_feats.index(c) for c in cat_cols if c in cb_feats]
    cb_probs = cb_model.predict_proba(Pool(X_cb, cat_features=cat_idx))[:, 1]

    stacking_probs = predict_meta_learner(
        meta_arts['model'], meta_arts['scaler'],
        {'xgb_prob': xgb_probs, 'lgbm_prob': lgbm_probs, 'catboost_prob': cb_probs}
    )
    inference_time = time.time() - t0

    # Metrics
    auc_roc = roc_auc_score(y_val, stacking_probs)
    auc_pr  = average_precision_score(y_val, stacking_probs)
    tps     = len(y_val) / inference_time

    decisions = apply_decision_thresholds(stacking_probs)
    dec_dist  = decisions['decision'].value_counts(normalize=True).to_dict()

    with mlflow.start_run(run_name='5e_batch_inference') as run:
        mlflow.log_params({
            'batch_size':  len(y_val),
            'models_used': 'XGBoost + LightGBM + CatBoost + Stacking',
        })
        mlflow.log_metrics({
            'val_auc_roc':         round(auc_roc, 5),
            'val_auc_pr':          round(auc_pr, 5),
            'inference_time_sec':  round(inference_time, 2),
            'throughput_tps':      round(tps, 0),
            **{f'decision_{k}': round(v, 4) for k, v in dec_dist.items()},
        })
        mlflow.set_tag('stage', 'batch_inference')

    print(f'Batch inference on {len(y_val):,} rows completed in {inference_time:.2f}s')
    print(f'Throughput: {tps:,.0f} tx/sec')
    print(f'AUC-ROC: {auc_roc:.5f}  |  AUC-PR: {auc_pr:.5f}')
    print(f'\nDecision distribution:')
    for k, v in dec_dist.items():
        print(f'  {k:<20}: {v:.1%}')

except FileNotFoundError as e:
    print(f'⚠️  Checkpoint missing: {e}. Run Exp 1-4 first.')

In [ ]:
# PR curve + decision distribution
try:
    prec, rec, _ = precision_recall_curve(y_val, stacking_probs)
    fpr, tpr, _  = roc_curve(y_val, stacking_probs)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ROC curve
    axes[0].plot(fpr, tpr, color='#3498db', lw=2, label=f'AUC-ROC={auc_roc:.4f}')
    axes[0].plot([0,1],[0,1],'--', color='gray', lw=1)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10)

    # PR curve
    axes[1].plot(rec, prec, color='#2ecc71', lw=2, label=f'AUC-PR={auc_pr:.4f}')
    axes[1].fill_between(rec, prec, alpha=0.1, color='#2ecc71')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10)

    # Decision distribution
    dec_counts = decisions['decision'].value_counts()
    dec_colors = {'auto_approve': '#2ecc71', 'flag_review': '#f39c12', 'block': '#e74c3c'}
    bars = axes[2].bar(
        dec_counts.index,
        dec_counts.values / len(decisions),
        color=[dec_colors.get(k, 'gray') for k in dec_counts.index],
        edgecolor='white'
    )
    for bar, val in zip(bars, dec_counts.values / len(decisions)):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
    axes[2].set_ylim(0, 1.1)
    axes[2].set_title('Decision Distribution', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('% of transactions')

    plt.suptitle('Fraud Detection — Batch Inference Results', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(str(SHAP_DIR / 'exp05_batch_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

    with mlflow.start_run(run_name='5e_batch_inference', nested=True):
        mlflow.log_artifact(str(SHAP_DIR / 'exp05_batch_results.png'))

except NameError:
    print('Run the batch inference cell first.')

---
## 5.6 — Training-Inference Parity Check

Runs 5 key feature columns through **both the training pipeline and the inference pipeline** and confirms they produce bit-identical values to 5 decimal places.

This is the single most important correctness test for a production ML system.

In [ ]:
# ── Run 5f: Training-inference parity ─────────────────────────────────────────
PARITY_COLS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'is_weekend', 'is_late_night',
    'D1_norm', 'D2_norm',
    'C1_C2_ratio', 'C5_C6_ratio', 'C6_C11_ratio', 'C3_flag',
    'M_match_count', 'M_mismatch_count', 'M_nan_count',
    'email_match', 'P_email_is_free', 'email_trusted',
    'device_mobile', 'device_os_encoded', 'device_browser_encoded',
]

tx = sample_transactions['legitimate'].copy()

# ── Training path ─────────────────────────────────────────────────────────────
train_df_pair = pd.DataFrame([tx, tx.copy()])
train_df_pair = encode_m_columns(train_df_pair, nan_value=-1)
train_df_pair = build_time_features(train_df_pair)
train_df_pair = build_d_features(train_df_pair, train_idx=None)
train_df_pair = build_c_features(train_df_pair, train_idx=None)
train_df_pair = build_m_features(train_df_pair, m_cols=None, train_idx=None)
train_df_pair = build_email_features(train_df_pair)
train_df_pair = build_device_features(train_df_pair)
device_encs   = get_device_encoders(train_df_pair)
train_df_pair = apply_device_encoders(train_df_pair, device_encs)
train_row     = train_df_pair.iloc[0]

# ── Inference path ────────────────────────────────────────────────────────────
infer_df = pd.DataFrame([tx.copy()])
infer_df = encode_m_columns(infer_df, nan_value=-1)
infer_df = build_time_features(infer_df)
infer_df = build_d_features(infer_df, train_idx=None)
infer_df = build_c_features(infer_df, train_idx=None)
infer_df = build_m_features(infer_df, m_cols=None, train_idx=None)
infer_df = build_email_features(infer_df)
infer_df = build_device_features(infer_df)
infer_df = apply_device_encoders(infer_df, device_encs)
infer_row = infer_df.iloc[0]

# ── Compare ───────────────────────────────────────────────────────────────────
parity_rows = []
n_pass = 0
n_fail = 0

for col in PARITY_COLS:
    if col not in train_row.index or col not in infer_row.index:
        parity_rows.append({'Feature': col, 'Train': 'N/A', 'Inference': 'N/A',
                             'Diff': 'N/A', 'Status': '⚠️ missing'})
        continue
    train_val = float(train_row[col])
    infer_val = float(infer_row[col])
    diff      = abs(train_val - infer_val)
    ok        = diff < 1e-5
    if ok: n_pass += 1
    else:  n_fail += 1
    parity_rows.append({
        'Feature':   col,
        'Train':     f'{train_val:.6f}',
        'Inference': f'{infer_val:.6f}',
        'Diff':      f'{diff:.2e}',
        'Status':    '✅ PASS' if ok else '❌ FAIL',
    })

parity_df = pd.DataFrame(parity_rows)

with mlflow.start_run(run_name='5f_parity_check') as run:
    mlflow.log_metrics({'parity_pass': n_pass, 'parity_fail': n_fail})
    mlflow.set_tag('stage', 'parity_check')
    mlflow.set_tag('all_passed', str(n_fail == 0))

print(parity_df.to_string(index=False))
print(f'\n{n_pass}/{n_pass + n_fail} features match to 5 decimal places')
if n_fail == 0:
    print('✅ TRAINING-INFERENCE PARITY: PASS — no feature drift detected')
else:
    print(f'❌ PARITY FAIL — {n_fail} features differ between training and inference pipeline!')

---
## 5.7 — Experiment Comparison Summary

Loads all `artifacts/metrics/expXX_*.json` files and renders the full progression from baseline to the production stack.

In [ ]:
# ── Run 5g: Cross-experiment summary ─────────────────────────────────────────
exp_records = []
for fn in sorted(METRICS_DIR.glob('exp0*.json')):
    try:
        with open(fn) as f:
            d = json.load(f)
        exp_records.append({
            'Experiment':  d.get('experiment', fn.stem),
            'AUC-ROC':     d.get('val_auc_roc', d.get('auc_roc', None)),
            'AUC-PR':      d.get('val_auc_pr', d.get('best_aupr', None)),
            'Description': d.get('model', d.get('description', '—')),
        })
    except Exception as e:
        print(f'Could not parse {fn.name}: {e}')

summary_df = pd.DataFrame(exp_records).dropna(subset=['AUC-PR'])
print(summary_df.to_string(index=False))

if len(summary_df) > 0:
    baseline_aupr = summary_df.iloc[0]['AUC-PR']
    summary_df['Δ Baseline'] = summary_df['AUC-PR'] - baseline_aupr

    with mlflow.start_run(run_name='5g_summary') as run:
        for _, row in summary_df.iterrows():
            mlflow.log_metrics({
                f"{row['Experiment']}_aupr":  round(float(row['AUC-PR']), 5),
                f"{row['Experiment']}_delta": round(float(row['Δ Baseline']), 5),
            })
        mlflow.set_tag('stage', 'summary')

In [ ]:
# Progress chart: AUC-PR across experiments
if len(summary_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))

    exps   = summary_df['Experiment'].tolist()
    auprs  = summary_df['AUC-PR'].tolist()
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(exps)))

    bars = ax.bar(exps, auprs, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, auprs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('AUC-PR (higher = better)', fontsize=11)
    ax.set_title('AUC-PR Progression Across All Experiments', fontsize=14, fontweight='bold')
    ax.set_ylim(min(auprs) * 0.95 if auprs else 0, max(auprs) * 1.08 if auprs else 1)
    ax.axhline(baseline_aupr, color='#e74c3c', ls='--', lw=1.5, label=f'Baseline ({baseline_aupr:.4f})')
    ax.legend(fontsize=10)
    plt.xticks(rotation=20, ha='right', fontsize=10)
    plt.tight_layout()
    plt.savefig(str(SHAP_DIR / 'exp05_progress_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()

    with mlflow.start_run(run_name='5g_summary', nested=True):
        mlflow.log_artifact(str(SHAP_DIR / 'exp05_progress_summary.png'))

    if len(summary_df) > 1:
        best_aupr  = summary_df['AUC-PR'].max()
        total_gain = best_aupr - baseline_aupr
        print(f'\n📈 Total AUC-PR improvement over baseline: +{total_gain:.5f}')
        print(f'   Baseline: {baseline_aupr:.5f} → Best: {best_aupr:.5f}')

print('\n✅ MLflow run complete. Open http://localhost:5000 to view all runs.')
print('   Experiment: 05_Pipeline_Walkthrough')